# Day 1 · Lab 2 — 다중 에이전트 공유 상태와 AgentCore Memory

**연계 모듈**: M1-3 메모리 및 상태 관리 <br>
**소요 시간**: 약 60분 · **생성 리소스**: AgentCore Memory 1개 (마지막에 삭제)

## 학습 목표
| 파트 | 내용 | 슬라이드 |
| --- | --- | --- |
| 1 | `invocation_state`로 LLM 컨텍스트 밖에서 상태 공유 | 35–36 |
| 2 | AgentCore Memory 단기 메모리를 여러 에이전트가 공유 | 30–33 |
| 3 | actor 단위 메모리 격리 확인 (보안 영향) | 25, 28 |
| 4 | 외부 상태 스토리지 설계 토론 | 37–38 |


## 0. 환경 설정

In [ ]:
%pip install -q -r requirements.txt

In [1]:
import boto3
import json
import time

REGION = boto3.Session().region_name or "us-west-2"
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
SUFFIX = time.strftime("%m%d%H%M%S")  # 리소스 이름 충돌 방지용

print(f"Region: {REGION} | Account: {ACCOUNT_ID} | Model: {MODEL_ID}")

from strands import Agent, tool, ToolContext
from strands.models import BedrockModel

model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2)

Region: us-east-1 | Account: 702995877146 | Model: global.anthropic.claude-haiku-4-5-20251001-v1:0


## 1. `invocation_state` — 모델이 보지 않는 공유 상태

고객 ID, 등급, 인증 토큰 같은 값은 **프롬프트에 넣지 않고** 도구에만 전달해야 합니다. <br>
`agent(..., invocation_state={...})` 로 넘긴 값은 `@tool(context=True)` 도구에서 `tool_context.invocation_state`로 읽을 수 있고, 스웜·그래프의 모든 노드로 자동 전파됩니다.

In [2]:
CUSTOMER_DB = {
    "C-001": {"name": "김민수", "tier": "GOLD", "orders": ["ORD-1001", "ORD-1002"]},
    "C-002": {"name": "이지은", "tier": "BASIC", "orders": ["ORD-1003"]},
}


@tool(context=True)
def get_my_orders(tool_context: ToolContext) -> str:
    """현재 로그인한 고객의 주문 목록을 조회합니다."""
    customer_id = tool_context.invocation_state.get("customer_id")
    if customer_id not in CUSTOMER_DB:
        return "인증된 고객 정보가 없습니다."
    return f"주문 목록: {', '.join(CUSTOMER_DB[customer_id]['orders'])}"


@tool(context=True)
def get_my_benefits(tool_context: ToolContext) -> str:
    """현재 로그인한 고객의 등급 혜택을 조회합니다."""
    tier = CUSTOMER_DB.get(tool_context.invocation_state.get("customer_id"), {}).get("tier", "UNKNOWN")
    benefits = {"GOLD": "무료 반품, 환불 수수료 면제", "BASIC": "30일 이내 반품"}
    return f"{tier} 등급 혜택: {benefits.get(tier, '없음')}"


agent = Agent(model=model, tools=[get_my_orders, get_my_benefits], callback_handler=None,
              system_prompt="고객 서비스 상담원입니다. 도구로 확인한 정보만 답하세요.")
result = agent("제 주문 목록이랑 받을 수 있는 혜택 알려주세요.", invocation_state={"customer_id": "C-001"})
print(result)

# 고객 ID가 모델 컨텍스트(메시지 기록)에 노출되었는지 확인
print("\n메시지 기록에 C-001 노출 여부:", "C-001" in json.dumps(agent.messages, ensure_ascii=False))

안녕하세요! 고객님의 정보를 확인했습니다.

**📦 주문 목록**
- ORD-1001
- ORD-1002

**🎁 GOLD 등급 혜택**
- 무료 반품
- 환불 수수료 면제

현재 GOLD 등급으로 무료 반품과 환불 수수료 면제 혜택을 받으실 수 있습니다. 추가로 궁금한 사항이 있으시면 말씀해주세요!


메시지 기록에 C-001 노출 여부: False


### 스웜 전체에 상태 전파
같은 `invocation_state`가 스웜의 모든 에이전트 도구로 전달됩니다. 에이전트 간 핸드오프 메시지에 고객 ID를 실어 보낼 필요가 없습니다.

In [3]:
from strands.multiagent import Swarm

orders_agent = Agent(name="orders", model=model, tools=[get_my_orders], callback_handler=None,
                     system_prompt="주문 담당. 주문 목록을 확인한 뒤 혜택 질문은 benefits에게 넘기세요.")
benefits_agent = Agent(name="benefits", model=model, tools=[get_my_benefits], callback_handler=None,
                       system_prompt="혜택 담당. 지금까지 정보를 종합해 고객에게 최종 답변하세요.")

swarm = Swarm([orders_agent, benefits_agent], entry_point=orders_agent, max_handoffs=4, execution_timeout=180)
sr = swarm("제 주문 목록과 반품 혜택을 알려주세요.", invocation_state={"customer_id": "C-002"})
print("경로:", " → ".join(n.node_id for n in sr.node_history))
print(sr.results[sr.node_history[-1].node_id].result)

경로: orders → benefits
고객님께 최종 답변을 드리겠습니다.

---

**📋 주문 정보 및 반품 혜택 안내**

**주문 번호:** ORD-1003

**현재 등급:** BASIC 등급

**반품 혜택:**
- ✅ **30일 이내 반품 가능**
  - 구매 후 30일 이내에 상품을 반품하실 수 있습니다.
  - 반품 시 배송료는 고객님 부담입니다.

**반품 절차:**
1. 고객센터에 반품 신청
2. 반품 배송 진행
3. 상품 수령 후 환불 처리

혹시 반품 절차에 대해 더 자세히 알고 싶으시거나 추가 문의사항이 있으시면 말씀해주세요!



## 2. AgentCore Memory로 에이전트 간 단기 메모리 공유

`invocation_state`는 **한 번의 호출 안에서만** 유효합니다. 여러 요청과 여러 에이전트에 걸쳐 대화를 이어가려면 외부 메모리가 필요합니다.

설계:
- 메모리 리소스 1개, 전략 없음(단기 메모리만)
- **actor_id = 고객**, **session_id = 상담 세션** 을 모든 전문가 에이전트가 공유
- 각 에이전트는 기록할 때 `[에이전트명]` 접두어를 붙여 누가 말했는지 남김

### 2-1. 메모리 리소스 생성 (약 1–2분)

In [4]:
from bedrock_agentcore.memory import MemoryClient

mem = MemoryClient(region_name=REGION)
memory = mem.create_memory_and_wait(
    name=f"Lab2SharedSTM_{SUFFIX}",
    description="Day1 Lab2 다중 에이전트 공유 단기 메모리",
    strategies=[],          # 단기 메모리만 사용
    event_expiry_days=7,
    max_wait=300,
    poll_interval=10,
)
MEMORY_ID = memory["id"]
print("Memory ID:", MEMORY_ID)

Memory ID: Lab2SharedSTM_1007025501-t7V0ez5xsC


### 2-2. 공유 메모리 훅
Strands 훅으로 에이전트 생명주기에 메모리 읽기·쓰기를 연결합니다.
- `AgentInitializedEvent`: 공유 세션의 최근 대화를 읽어 시스템 프롬프트에 추가
- `MessageAddedEvent`: 새 텍스트 메시지를 공유 세션에 기록

In [5]:
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent


class SharedMemoryHook(HookProvider):
    def __init__(self, label: str, actor_id: str, session_id: str, k: int = 10):
        self.label, self.actor_id, self.session_id, self.k = label, actor_id, session_id, k

    def on_init(self, event: AgentInitializedEvent):
        turns = mem.get_last_k_turns(memory_id=MEMORY_ID, actor_id=self.actor_id,
                                     session_id=self.session_id, k=self.k)
        lines = [f"{m['role']}: {m['content']['text']}" for turn in turns for m in turn]
        if lines:
            event.agent.system_prompt += "\n\n[공유 상담 기록]\n" + "\n".join(lines)
        print(f"   ({self.label}) 공유 기록 {len(lines)}건 로드")

    def on_message(self, event: MessageAddedEvent):
        msg = event.message
        texts = [c["text"] for c in msg["content"] if "text" in c]
        if not texts:
            return  # toolUse/toolResult 블록은 기록하지 않음
        role = "USER" if msg["role"] == "user" else "ASSISTANT"
        mem.create_event(memory_id=MEMORY_ID, actor_id=self.actor_id, session_id=self.session_id,
                         messages=[(f"[{self.label}] {' '.join(texts)}", role)])

    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(AgentInitializedEvent, self.on_init)
        registry.add_callback(MessageAddedEvent, self.on_message)

### 2-3. 메모리를 공유하는 전문가 에이전트
`make_specialist_tools(actor, session)`는 해당 고객·세션에 묶인 전문가 도구 세트를 만듭니다.

In [6]:
ORDERS = {
    "ORD-1001": {"item": "노트북 Pro 14", "status": "DELIVERED"},
    "ORD-1002": {"item": "무선 이어폰", "status": "SHIPPING", "eta": "10월 9일"},
}


@tool
def lookup_order(order_id: str) -> str:
    """주문 상세를 조회합니다."""
    return json.dumps(ORDERS.get(order_id, {}), ensure_ascii=False)


def make_specialist_tools(actor_id: str, session_id: str):
    @tool
    def order_agent(query: str) -> str:
        """주문 상세·상태 문의를 처리하는 주문 전문가입니다."""
        a = Agent(model=model, tools=[lookup_order], callback_handler=None,
                  system_prompt="주문 전문가입니다. 도구로 확인한 내용만 답하세요.",
                  hooks=[SharedMemoryHook("order-agent", actor_id, session_id)])
        return str(a(query))

    @tool
    def shipping_agent(query: str) -> str:
        """배송 일정 문의를 처리하는 배송 전문가입니다. 이전 상담 기록을 참고합니다."""
        a = Agent(model=model, tools=[lookup_order], callback_handler=None,
                  system_prompt="배송 전문가입니다. 공유 상담 기록에서 언급된 주문을 우선 참고하세요.",
                  hooks=[SharedMemoryHook("shipping-agent", actor_id, session_id)])
        return str(a(query))

    return [order_agent, shipping_agent]


def new_orchestrator(actor_id, session_id):
    return Agent(model=model, tools=make_specialist_tools(actor_id, session_id), callback_handler=None,
                 system_prompt="고객 서비스 오케스트레이터. 주문 문의는 order_agent, 배송 문의는 shipping_agent에 위임하세요. "
                               "전문가에게 넘길 때 고객 원문을 그대로 전달하세요.")


CUSTOMER = "customer-C001"
SESSION = f"support-session-{SUFFIX}"

### 2-4. 공유 메모리 동작 확인
1. 첫 번째 오케스트레이터가 주문 전문가에게 **이어폰 주문(ORD-1002)** 을 확인시킵니다.
2. 오케스트레이터를 **새로 생성**(프로세스 재시작 가정)하고 "아까 그 이어폰"이라고만 묻습니다.
3. 배송 전문가는 주문 ID를 듣지 못했지만 공유 메모리에서 ORD-1002를 찾아야 합니다.

In [7]:
orch1 = new_orchestrator(CUSTOMER, SESSION)
print(orch1("ORD-1002 주문이 뭐였는지 확인해 주세요."))

   (order-agent) 공유 기록 0건 로드
ORD-1002 주문 정보입니다:

- **상품**: 무선 이어폰
- **상태**: 배송 중 (SHIPPING)
- **예상 도착일**: 10월 9일

현재 배송 중이며, 10월 9일에 도착할 예정입니다. 다른 문의사항이 있으시면 말씀해주세요!



In [8]:
time.sleep(3)  # 이벤트 저장 반영 대기
orch2 = new_orchestrator(CUSTOMER, SESSION)  # 이전 대화 기록이 없는 새 오케스트레이터
print(orch2("아까 확인한 그 이어폰 언제 도착해요?"))

   (shipping-agent) 공유 기록 2건 로드
안녕하세요! 확인해드렸습니다.

**ORD-1002 주문의 무선 이어폰은 10월 9일에 도착할 예정**입니다.

현재 배송 중 상태이므로 예정된 날짜에 받으실 수 있을 것으로 보입니다. 

혹시 배송 상태에 변동이 있거나 다른 문의사항이 있으시면 언제든지 말씀해 주세요!



In [9]:
# 공유 세션에 누가 무엇을 기록했는지 타임라인 확인
turns = mem.get_last_k_turns(memory_id=MEMORY_ID, actor_id=CUSTOMER, session_id=SESSION, k=20)
for turn in turns:
    for m in turn:
        print(f"{m['role']:<10} {m['content']['text'][:110]}")

ASSISTANT  [shipping-agent] 공유 상담 기록에 따르면, ORD-1002 주문의 **무선 이어폰**은 **10월 9일**에 도착할 예정입니다.

현재 배송 중(SHIPPING) 상태이므로, 예정된 
USER       [shipping-agent] 아까 확인한 그 이어폰 언제 도착해요?
ASSISTANT  [order-agent] ORD-1002 주문 정보입니다:

- **상품**: 무선 이어폰
- **상태**: 배송 중 (SHIPPING)
- **예상 도착일**: 10월 9일

현재 배송 중이며, 
USER       [order-agent] ORD-1002 주문이 뭐였는지 확인해 주세요.


## 3. 메모리 격리 확인

다른 고객(actor)의 세션은 서로 보이지 않아야 합니다. 같은 세션 ID를 쓰더라도 actor가 다르면 기록이 분리됩니다.
프로덕션에서는 여기에 **IAM 조건 키**로 actor 단위 접근을 강제합니다 (Day 2 보안 모듈과 연결).

In [10]:
other = mem.get_last_k_turns(memory_id=MEMORY_ID, actor_id="customer-C002", session_id=SESSION, k=20)
print("customer-C002가 같은 세션 ID로 조회한 기록 수:", sum(len(t) for t in other))

customer-C002가 같은 세션 ID로 조회한 기록 수: 0


## 4. 토론: 어떤 상태를 어디에 둘 것인가

| 상태 유형 | 예시 | 권장 위치 | 이유 |
| --- | --- | --- | --- |
| 요청 범위 컨텍스트 | 고객 ID, 인증 토큰 | `invocation_state` | 모델에 노출되지 않음 |
| 대화 기록 | 상담 턴 | AgentCore Memory 단기 | 세션 간 지속, 에이전트 공유 |
| 사실·선호 | "ThinkPad 선호" | AgentCore Memory 장기 (Lab 3) | 의미 검색으로 선택 주입 |
| 트랜잭션 데이터 | 주문, 환불 원장 | 외부 DB (DynamoDB 등) | 정합성·감사 요구 |

1. 의료 레코드 에이전트(슬라이드 37)라면 위 표에서 무엇이 달라져야 할까요?
2. 공유 메모리에 에이전트가 잘못된 사실을 쓰면 다른 에이전트가 오염됩니다(컨텍스트 포이즈닝). 어떻게 막겠습니까?

## 5. 리소스 정리

In [ ]:
mem.delete_memory_and_wait(memory_id=MEMORY_ID, max_wait=300, poll_interval=10)
print("삭제 완료:", MEMORY_ID)